# Met Eyes Experiments

# Get Data

In [ ]:
import json
import requests

from os import listdir, makedirs, path
from PIL import Image as PImage
from time import sleep

from utils import export_combined_jsons

DATA_DIR = "./lehman-data"
IMG_DIR = f"{DATA_DIR}/image"
JSON_DIR = f"{DATA_DIR}/json"

JSON_OBJS_DIR = f"{JSON_DIR}/objects"

## The Met API

https://metmuseum.github.io/

https://github.com/metmuseum/openaccess

**Please limit request rate to 80 requests per second.**

In [ ]:
makedirs(IMG_DIR, exist_ok=True)
makedirs(JSON_DIR, exist_ok=True)
makedirs(JSON_OBJS_DIR, exist_ok=True)

In [ ]:
MET_URL = "https://collectionapi.metmuseum.org/public/collection/v1"

SEARCH_DEPARTMENT_IDS = []
SEARCH_DEPARTMENTS = ["Robert Lehman", "Armor"][:1]
SEARCH_MEDIUMS = ["Paintings", "Drawings"][:1]

### Get Department IDs

In [ ]:
dept_response = requests.get(f"{MET_URL}/departments")
dept_data = dept_response.json()["departments"]

dept_name2id = { d["displayName"] : d["departmentId"] for d in dept_data }

for sdpt in SEARCH_DEPARTMENTS:
  for dname,did in dept_name2id.items():
    if sdpt.lower() in dname.lower():
      SEARCH_DEPARTMENT_IDS.append(did)

### Get Object IDs

In [ ]:
obj_ids = []

for dpt_query in SEARCH_DEPARTMENT_IDS:
  for medium_query in SEARCH_MEDIUMS:
    collection_response = requests.get(f"{MET_URL}/search?medium={medium_query}&departmentId={dpt_query}&q=*")
    query_obj_ids = set(collection_response.json()["objectIDs"])
    obj_ids += list(query_obj_ids)

len(obj_ids)

### Get Object Metadata

In [ ]:
obj_fields = ["objectID", "objectName", "title", "primaryImage", "primaryImageSmall", "artistRole", "artistDisplayName"]
obj_files = sorted(f for f in listdir(JSON_OBJS_DIR) if f.endswith("json"))

for cnt,oid in enumerate(obj_ids):
  if cnt % 20 == 0:
    print(f"{cnt} / {len(obj_ids)}")

  obj_json_path = f"{JSON_OBJS_DIR}/{oid}.json"
  if f"{oid}.json" in obj_files:
    continue

  obj_response = requests.get(f"{MET_URL}/objects/{oid}")
  obj_data = obj_response.json()
  obj_filtered_data = { f: obj_data[f] for f in obj_fields }

  obj_json_path = f"{JSON_OBJS_DIR}/{oid}.json"
  with open(obj_json_path, "w") as ofp:
    json.dump(obj_filtered_data, ofp)
  sleep(0.333)

### Export Combined Object Metadata

In [ ]:
export_combined_jsons(f"{JSON_DIR}/objects", JSON_DIR, "objects")

### Get Images

In [ ]:
for size in ["original", "900", "500"]:
  makedirs(f"{IMG_DIR}/{size}/", exist_ok=True)

with open(f"{JSON_DIR}/objects.json", "r") as ifp:
  obj_data = json.load(ifp)["objects"]

for cnt,obj in enumerate(obj_data):
  if cnt % 20 == 0:
    print(f"{cnt} / {len(obj_data)}")

  img_url = obj["primaryImage"]
  if not (img_url and len(img_url) > 0):
    continue

  oid = obj["objectID"]

  img_orig_path = f"{IMG_DIR}/original/{oid}.jpg"
  img_900_path = f"{IMG_DIR}/900/{oid}.jpg"
  img_500_path = f"{IMG_DIR}/500/{oid}.jpg"
  if path.isfile(img_orig_path) and path.isfile(img_900_path) and path.isfile(img_500_path):
    continue

  img_response = requests.get(img_url, stream=True)
  img = PImage.open(img_response.raw)

  if not path.isfile(img_orig_path):
    img.save(img_orig_path)

  if not path.isfile(img_900_path):
    img.thumbnail((900, 900))
    img.save(img_900_path)

  if not path.isfile(img_500_path):
    img.thumbnail((500, 500))
    img.save(img_500_path)

  sleep(0.333)